# Transformer 与 Vision Transformer

注意力用查询 Q 与键 K 的相似度为值 V 加权。计算 QKᵀ 后除以每个头的维度平方根，并沿键的维度做 softmax。掩码可以屏蔽填充或未来位置。多头注意力把投影后的特征拆成多个头，每个头独立计算，再拼接投影。

[Transformer 结构图](../../assets/transformer.png) 展示了编码器和解码器。编码器的 Q、K、V 来自同一序列，解码器的交叉注意力用解码器状态作 Q、编码器输出作 K 和 V。

参考 [Attention Is All You Need](https://arxiv.org/abs/1706.03762) 和 [An Image Is Worth 16×16 Words](https://arxiv.org/abs/2010.11929)。

In [ ]:
import torch
from torch import nn


def scaled_dot_product(query, key, value, mask=None):
    logits = query @ key.transpose(-2, -1) / query.shape[-1] ** 0.5
    if mask is not None:
        logits = logits.masked_fill(mask, float("-inf"))
        logits = logits.masked_fill(mask.all(dim=-1, keepdim=True), 0.0)
    weights = logits.softmax(dim=-1)
    if mask is not None:
        weights = weights.masked_fill(mask, 0.0)
    return weights @ value, weights


class MultiHeadAttention(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        if heads <= 0 or dim % heads:
            raise ValueError("特征维度必须能被头数整除")
        self.heads = heads
        self.head_dim = dim // heads
        self.query = nn.Linear(dim, dim)
        self.key = nn.Linear(dim, dim)
        self.value = nn.Linear(dim, dim)
        self.output = nn.Linear(dim, dim)

    def split(self, tokens):
        batch, length, _ = tokens.shape
        return tokens.reshape(batch, length, self.heads, self.head_dim).transpose(1, 2)

    def forward(self, query, key, value, mask=None):
        attended, _ = scaled_dot_product(
            self.split(self.query(query)), self.split(self.key(key)),
            self.split(self.value(value)), mask,
        )
        return self.output(attended.transpose(1, 2).reshape_as(query))


torch.manual_seed(0)
q, k, v = torch.randn(2, 3, 32), torch.randn(2, 5, 32), torch.randn(2, 5, 32)
mask = torch.zeros(3, 5, dtype=torch.bool)
mask[:, -1] = True
out, weights = scaled_dot_product(q, k, v, mask)
assert out.shape == (2, 3, 32)
assert torch.all(weights[..., -1] == 0)
assert torch.allclose(weights.sum(-1), torch.ones(2, 3))
assert MultiHeadAttention(32, 4)(q, k, v, mask).shape == q.shape
assert scaled_dot_product(q, k, v, torch.ones_like(mask))[0].count_nonzero() == 0

## ViT

步长与卷积核大小都等于 patch 尺寸，将图像划成不重叠的块，并映射为 token。添加类别 token 和可学习的位置编码，再通过预归一化的编码块。LayerNorm 在每个 token 的特征维度上归一化，参数数量由特征维度决定。最后用类别 token 预测分类 logits。

In [ ]:
def layer_norm(tokens, weight, bias, eps=1e-5):
    mean = tokens.mean(dim=-1, keepdim=True)
    variance = tokens.var(dim=-1, correction=0, keepdim=True)
    return (tokens - mean) * torch.rsqrt(variance + eps) * weight + bias


norm = nn.LayerNorm(q.shape[-1])
assert torch.allclose(layer_norm(q, norm.weight, norm.bias), norm(q), atol=1e-6)

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.attention = MultiHeadAttention(dim, heads)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))

    def forward(self, tokens):
        normalized = self.norm1(tokens)
        tokens = tokens + self.attention(normalized, normalized, normalized)
        return tokens + self.mlp(self.norm2(tokens))


class VisionTransformer(nn.Module):
    def __init__(self, image_size, patch_size, num_classes, dim, depth, heads):
        super().__init__()
        if patch_size <= 0 or image_size <= 0 or image_size % patch_size:
            raise ValueError("图像尺寸必须是 patch 尺寸的倍数")
        self.image_size = image_size
        self.patch_embed = nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        num_patches = (image_size // patch_size) ** 2
        self.positions = nn.Parameter(torch.zeros(1, num_patches + 1, dim))
        self.blocks = nn.Sequential(*(EncoderBlock(dim, heads) for _ in range(depth)))
        self.classifier = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, num_classes))

    def forward(self, image):
        if image.shape[-2:] != (self.image_size, self.image_size):
            raise ValueError("输入图像尺寸与模型配置不一致")
        patches = self.patch_embed(image).flatten(2).transpose(1, 2)
        tokens = torch.cat((self.cls_token.expand(image.shape[0], -1, -1), patches), dim=1)
        encoded = self.blocks(tokens + self.positions)
        return self.classifier(encoded[:, 0])


model = VisionTransformer(image_size=32, patch_size=8, num_classes=10, dim=32, depth=2, heads=4)
logits = model(torch.randn(2, 3, 32, 32))
assert logits.shape == (2, 10)
logits.square().mean().backward()
assert model.patch_embed.weight.grad is not None